In [ ]:
# -*- coding: utf-8 -*-
"""
BLG447 Görüntü İşleme – Final Projesi
Knee Osteoarthritis X-ray Texture Analysis

Amaç:
- Diz X-ray görüntülerinden GLCM tabanlı doku özellikleri çıkarmak
- OA / no_OA binary sınıflandırması için veri hazırlamak

Kullanılan Özellikler:
- GLCM (çoklu açı, tek mesafe)
"""

# =========================
# GEREKLİ KÜTÜPHANELER
# =========================
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import MinMaxScaler


# =========================
# SINIF HARİTALAMA
# =========================
def map_kl_to_binary(folder_name):
    """
    Binary sınıflandırma:
    0  -> no_OA
    1,2,3,4 -> OA
    """
    if folder_name == "0":
        return "no_OA"
    else:
        return "OA"


# =========================
# SOFT ROI
# =========================
def soft_roi(image):
    """
    Eklem bölgesini koruyan, kenar gürültüsünü azaltan soft ROI
    """
    h, w = image.shape[:2]

    y1 = int(h * 0.15)
    y2 = int(h * 0.90)
    x1 = int(w * 0.15)
    x2 = int(w * 0.85)

    return image[y1:y2, x1:x2]


# =========================
# PREPROCESSING
# =========================
def preprocess_image(image):
    roi = soft_roi(image)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    gray = cv2.resize(gray, (128, 128))
    return gray


# =========================
# GLCM QUANTIZATION
# =========================
def quantize(gray, levels=32):
    """
    Gri seviye değerlerini GLCM için düşürür
    """
    return (gray / 256 * levels).astype(np.uint8)


# =========================
# GLCM FEATURE EXTRACTION
# =========================
def extract_glcm_features(gray):
    """
    Çoklu açı + tek mesafe GLCM özellikleri
    """
    gray_q = quantize(gray, levels=32)

    distances = [32]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]

    glcm = graycomatrix(
        gray_q,
        distances=distances,
        angles=angles,
        levels=32,
        symmetric=True,
        normed=True
    )

    features = {}
    props = [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM"
    ]

    for prop in props:
        features[f"glcm_{prop}"] = np.mean(graycoprops(glcm, prop))

    return features


# =========================
# DATASET OKUMA
# =========================
def process_dataset(dataset_path):
    data = []
    splits = ["train", "val", "test"]

    for split in splits:
        split_path = os.path.join(dataset_path, split)
        if not os.path.isdir(split_path):
            continue
        
        for class_name in sorted(os.listdir(split_path)):
            class_folder = os.path.join(split_path, class_name)
            if not os.path.isdir(class_folder):
                continue

            mapped_class = map_kl_to_binary(class_name)
            if mapped_class is None:
                continue

            print(f"{split}/{class_name} -> {mapped_class}")
            
            for file_idx, file in enumerate(os.listdir(class_folder)):
                if file_idx % 50 == 0:
                    print(f"{split}/{class_name}: {file_idx} images processed")

                if file.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tiff")):
                    img_path = os.path.join(class_folder, file)
                    img = cv2.imread(img_path)
                    if img is None:
                        continue

                    gray = preprocess_image(img)

                    features = {}
                    features.update(extract_glcm_features(gray))
                    features["class"] = mapped_class
                    features["split"] = split
                    data.append(features)

    return pd.DataFrame(data)


# =========================
# NORMALIZATION
# =========================
def apply_minmax_scaling(df):
    feature_cols = df.select_dtypes(include=[np.number]).columns
    scaler = MinMaxScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])
    return df


# =========================
# ARFF KAYDETME
# =========================
def save_to_arff(df, filename):
    with open(filename, "w") as f:
        f.write("@RELATION knee_OA_GLCM_SOFT_ROI\n\n")

        for col in df.columns[:-1]:
            f.write(f"@ATTRIBUTE {col} NUMERIC\n")

        classes = ",".join(sorted(df["class"].unique()))
        f.write(f"@ATTRIBUTE class {{{classes}}}\n\n@DATA\n")

        for _, row in df.iterrows():
            f.write(",".join(map(str, row.values)) + "\n")

    print("✔ ARFF dosyası oluşturuldu:", filename)


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    dataset_path = r"C:\Users\BBN\Desktop\goruntuisleme\KneeOsteoarthritis\Knee_OA_Dataset"

    df = process_dataset(dataset_path)
    df = apply_minmax_scaling(df)
    save_to_arff(df, "GLCM_ONLY_SOFT_ROI_DİSTANCE32.arff")


0 -> no_OA
0: 0 görüntü işlendi
0: 50 görüntü işlendi
0: 100 görüntü işlendi
0: 150 görüntü işlendi
0: 200 görüntü işlendi
0: 250 görüntü işlendi
0: 300 görüntü işlendi
0: 350 görüntü işlendi
0: 400 görüntü işlendi
0: 450 görüntü işlendi
0: 500 görüntü işlendi
0: 550 görüntü işlendi
0: 600 görüntü işlendi
0: 650 görüntü işlendi
0: 700 görüntü işlendi
0: 750 görüntü işlendi
0: 800 görüntü işlendi
0: 850 görüntü işlendi
0: 900 görüntü işlendi
0: 950 görüntü işlendi
0: 1000 görüntü işlendi
0: 1050 görüntü işlendi
0: 1100 görüntü işlendi
0: 1150 görüntü işlendi
0: 1200 görüntü işlendi
0: 1250 görüntü işlendi
0: 1300 görüntü işlendi
0: 1350 görüntü işlendi
0: 1400 görüntü işlendi
0: 1450 görüntü işlendi
0: 1500 görüntü işlendi
0: 1550 görüntü işlendi
0: 1600 görüntü işlendi
0: 1650 görüntü işlendi
0: 1700 görüntü işlendi
0: 1750 görüntü işlendi
0: 1800 görüntü işlendi
0: 1850 görüntü işlendi
0: 1900 görüntü işlendi
0: 1950 görüntü işlendi
0: 2000 görüntü işlendi
0: 2050 görüntü işlendi
0: 2